# L5B15 Decision Data — EDA

Exploratory analysis of the processed L5B15 scoring events.
Reads from `data/processed/l5b15_decisions.parquet` — run `02_data_ingestion.ipynb` first.

In [ ]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("Imports OK")

In [ ]:
PROCESSED_FILE = Path("../data/processed/l5b15_decisions.parquet")
df_model = pd.read_parquet(PROCESSED_FILE)
print(f"Loaded {len(df_model):,} rows, {df_model.shape[1]} columns")
df_model.head(3)

## Data Quality Assessment

Assess missingness, constant columns, target distribution, feature correlations, and categorical cardinality.

In [ ]:
### 6.1 Feature Prefix Analysis
prefix_counts = Counter()

for c in df_model.columns:
    if "::" in c:
        prefix = c.split("::")[0]
    elif "." in c:
        prefix = c.split(".")[0]
    else:
        prefix = "(no prefix)"
    prefix_counts[prefix] += 1

prefix_counts

In [ ]:
### 6.2 Missingness and Constants
summary = pd.DataFrame({
    "dtype": df_model.dtypes.astype(str),
    "missing_frac": df_model.isna().mean(),
    "n_unique": df_model.nunique(dropna=True)
}).sort_values(["missing_frac", "n_unique"], ascending=[False, True])

summary.head(50)

In [ ]:
#Are there columns that are mostly missing?
mostly_missing = summary[summary["missing_frac"] > 0.95]
mostly_missing

In [ ]:
# constants check
constant_cols = summary[summary["n_unique"] <= 1]
constant_cols

In [ ]:
### 6.3 Target Distribution


plt.figure(figsize=(8, 4))
plt.hist(df_model["propensity"], bins=60)
plt.xlabel("L5B15 propensity")
plt.ylabel("Count")
plt.title("Distribution of L5B15 propensity")
plt.show()

In [ ]:
### 6.4 Feature Correlation Analysis
df_num_try = df_model.copy()

for col in df_num_try.columns:
    if df_num_try[col].dtype == "object":
        df_num_try[col] = pd.to_numeric(df_num_try[col], errors="ignore")

num_cols = df_num_try.select_dtypes(include="number").columns.tolist()
num_cols = [c for c in num_cols if c != "propensity"]

corr = df_num_try[num_cols + ["propensity"]].corr(numeric_only=True)["propensity"].drop("propensity")
corr.abs().sort_values(ascending=False).head(30)

In [ ]:
### 6.5 Categorical Feature Cardinality
cat_summary = pd.DataFrame({
    "n_unique": df_model.select_dtypes(include="object").nunique(dropna=True)
}).sort_values("n_unique", ascending=False)

cat_summary.head(30)